In [2]:
import pandas as pd
import numpy as np
import math
from collections import Counter
import re
import textstat

def tokenize(text): #this tokenizer will split words based on the aplabetical content, 
    if not text: # "def tokenize(text): boom1 bang" -> print(tokenize("def tokenize(text): boom1 bang"))
        return []
    return re.findall(r"[A-Za-z_][A-Za-z0-9_]*", text.lower())

def calculate_entropy(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return np.nan 
    words = tokenize(text)
    if not words:
        return np.nan
    counts = Counter(words)
    probs = [c / len(words) for c in counts.values()]
    return -sum(p * math.log2(p) for p in probs)

# --- Tree-sitter based strip_comments (matches dataset/buildDataset/build.py) ---
# pip install tree-sitter tree-sitter-language-pack
from tree_sitter import Parser as _TSParser
from tree_sitter_language_pack import get_language as _ts_get_language

_TS_LANGUAGE_MAP = {
    '.py': 'python',
    '.js': 'javascript', '.jsx': 'javascript',
    '.java': 'java',
    '.c': 'c', '.h': 'c',
    '.cpp': 'cpp', '.cc': 'cpp', '.hpp': 'cpp', '.cxx': 'cpp', '.hxx': 'cpp',
    '.cs': 'csharp',
    '.go': 'go',
    '.kt': 'kotlin', '.kts': 'kotlin',
    '.php': 'php',
    '.scala': 'scala',
    '.swift': 'swift',
}
_ts_parser_cache = {}


def _get_ts_parser(file_extension):
    lang_name = _TS_LANGUAGE_MAP.get(file_extension.lower())
    if lang_name is None:
        return None
    if lang_name not in _ts_parser_cache:
        _ts_parser_cache[lang_name] = _TSParser(_ts_get_language(lang_name))
    return _ts_parser_cache[lang_name]


def strip_comments(text, file_extension=None):
    """
    Returns the source with all comments/docstrings removed, using a
    tree-sitter parse instead of regex. The old regex version never
    removed a multi-line docstring (only same-line \"\"\"...\"\"\" was),
    blindly treated any line starting with '#' as a comment (wrongly
    deleting C preprocessor directives or Swift '#' macro calls), and
    truncated a line at a '//' inside a string literal (e.g. a URL).
    """
    ext = file_extension.lower() if file_extension else ""
    parser = _get_ts_parser(ext)
    if parser is None:
        return text

    parse_text = "<?php\n" + text if ext == ".php" else text
    parse_bytes = parse_text.encode("utf-8")
    tree = parser.parse(parse_bytes)

    remove_ranges = []

    def visit(node):
        if "comment" in node.type:
            remove_ranges.append((node.start_byte, node.end_byte))
            return
        if (ext == ".py" and node.type == "string"
                and node.parent is not None and node.parent.type in ("module", "block")):
            remove_ranges.append((node.start_byte, node.end_byte))
            return
        for child in node.children:
            visit(child)

    visit(tree.root_node)
    remove_ranges.sort()

    kept = bytearray()
    cursor = 0
    for start, end in remove_ranges:
        kept += parse_bytes[cursor:start]
        cursor = end
    kept += parse_bytes[cursor:]

    result = kept.decode("utf-8", errors="replace")
    if ext == ".php":
        result = result[len("<?php\n"):]

    clean_lines = [line.rstrip() for line in result.splitlines() if line.strip()]
    return "\n".join(clean_lines)


def _file_extension(file_path):
    file_path = str(file_path)
    return "." + file_path.rsplit(".", 1)[-1].lower() if "." in file_path else ""

def doc_code_overlap(doc_text, code_text): # get percent of tokens that overlap between code and text
    doc_tokens = set(tokenize(doc_text))
    code_tokens = set(tokenize(code_text))

    if not doc_tokens or not code_tokens:
        return np.nan
    overlap = doc_tokens.intersection(code_tokens)
    return len(overlap) / len(doc_tokens)

def doc_redundancy(doc_text): # get percent of tokens that are repeated within the documentation
    tokens = tokenize(doc_text)
    if not tokens:
        return np.nan
    
    unique = len(set(tokens))
    return 1 - (unique / len(tokens))

In [3]:
import pandas as pd
import numpy as np
import math
from collections import Counter
import re
import textstat

output_path = r"G:\On the Naturalness of Agent-Generated Documentation\dataset\data\dev_agent_combined.csv"
df = pd.read_csv(r"G:\On the Naturalness of Agent-Generated Documentation\dataset\data\agent_dataset_subset_data.csv")
df_human = pd.read_csv(r"G:\On the Naturalness of Agent-Generated Documentation\dataset\data\dev_dataset_subset_data.csv")

df["group"] = "agent"
df_human["group"] = "human"
df = pd.concat([df, df_human], ignore_index=True)


def clean_comments(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # Single alternation scanned left-to-right (finditer) instead of three
    # independent findall passes over the same text. Three separate passes
    # can double-count: if a "#"/"//" comment's content happens to contain
    # something that looks like a \"\"\"...\"\"\" span (or vice versa), the
    # old code matched it in both passes and appended it twice. The inline
    # alternative also stops at a following triple-quote or /* block, not
    # just the next #/// or end of string -- otherwise a "#" comment right
    # next to a standalone docstring fragment swallows it whole, quote
    # markers included, instead of letting it be independently matched
    # and stripped.
    combined_pattern = re.compile(
        r'/\*+([\s\S]*?)\*/'
        r'|["\']{3}([\s\S]*?)["\']{3}'
        r'|(?:#|///|//)\s*(.*?)\s*(?=#|///|//|["\']{3}|/\*|$)'
    )

    cleaned_comments = []
    for m in combined_pattern.finditer(text):
        item = next(g for g in m.groups() if g is not None)

        # Remove the leading '*' from each line (common in JSDoc/C-style)
        clean_item = re.sub(r'^\s*\* ?', '', item, flags=re.MULTILINE)

        # Collapse newlines and tabs into a single space
        clean_item = ' '.join(clean_item.split())

        if clean_item.strip():
            cleaned_comments.append(clean_item.strip())

    return ' '.join(cleaned_comments)


# Output: ['This is a docstring', 'This is an inline block', 'This is a single line comment', 'Another comment here']

documentation = {}

for i,a in zip(df['doc_text'], df['label']):
    if pd.isna(a):
        a = 'human'
    if a in documentation:
        documentation[a].append(clean_comments(i))
    else:
        documentation[a] = [clean_comments(i)]


for idx, row in df.iterrows():
    print(row["doc_text"])
    doc_text = clean_comments(row["doc_text"])
    print(doc_text)
    print("\n\n-------------------------\n\n")

    code_text = row["function"]
    file_extension = _file_extension(row["file_path"])
    code_text_no_doc = strip_comments(code_text, file_extension)
    if isinstance(doc_text, str) and doc_text.strip():
        
        df.at[idx, "doc_entropy"] = calculate_entropy(doc_text)
        df.at[idx, "doc_readability"] = textstat.flesch_reading_ease(doc_text) if doc_text else np.nan
        df.at[idx, "doc_code_overlap"] = doc_code_overlap(doc_text, code_text_no_doc)
        df.at[idx, "doc_redundancy"] = doc_redundancy(doc_text)
    else:
        # For missing or empty docs, keep NaN
        df.at[idx, "doc_entropy"] = np.nan
        df.at[idx, "doc_readability"] = np.nan
        df.at[idx, "doc_code_overlap"] = np.nan
        df.at[idx, "doc_redundancy"] = np.nan


# Save the updated dataset

df.to_csv(output_path, index=False)
print("Recomputed metrics and saved to updated_dataset_2_metrics.csv")

df = pd.read_csv(output_path)
df = df[~df['file_path'].str.contains(r'\.next/', na=False)]
df.to_csv(output_path, index=False)

nan



-------------------------


// Load first chapter by default
Load first chapter by default


-------------------------


// HTML files to update // Add premium features script before closing body tag // Add data-transition attributes to chapter links // Add magnetic element classes // Create test page for premium features // Run integration
HTML files to update Add premium features script before closing body tag Add data-transition attributes to chapter links Add magnetic element classes Create test page for premium features Run integration


-------------------------


nan



-------------------------


// Check for reduced motion preference
Check for reduced motion preference


-------------------------


// Add data-transition attribute to chapter links
Add data-transition attribute to chapter links


-------------------------


// Add data-transition attribute to chapter links
Add data-transition attribute to chapter links


-------------------------


// Add magnetic proper

## Deduplicate

`build.py`'s mining pipeline has no check for "was this PR already
successfully processed and written before" -- it opens the output CSV in
append mode and reprocesses every PR in the input parquet every run. If a
mining run is ever interrupted and resumed (e.g. after re-cloning repos to
retry previously-failed clones), any PR that had already succeeded and been
written gets reprocessed and appended a second time, producing exact
duplicate rows (same repo/PR/file/function/start_line, identical content).
Confirmed present in both `agent_final_dataset_subset_b.csv` (109 duplicate
groups) and newer mining runs (115 groups, ~3% of rows) -- dedupe
defensively here before any further merging or metric computation.

In [4]:
before = len(df)
df = df.drop_duplicates(
    subset=['repo', 'pull_request', 'file_path', 'function_name', 'function_start_line'],
    keep='first'
).reset_index(drop=True)
removed = before - len(df)

if removed:
    print(f'Removed {removed} duplicate rows (same repo/PR/file/function/start_line).')
else:
    print('No duplicate rows found.')

df.to_csv(output_path, index=False)


Removed 185 duplicate rows (same repo/PR/file/function/start_line).


## If not merged already, merge in the new agent data

In [ ]:
# ── 9. Merge new_agent_dataset.csv + final_dataset.csv → final_dataset_new.csv ──
NEW_AGENT_OUTPUT_PATH = r"G:\On the Naturalness of Agent-Generated Documentation\dataset\data\agent_supplement_dataset.csv"


df_original = pd.read_csv(output_path)
df_new      = pd.read_csv(NEW_AGENT_OUTPUT_PATH)

print(f'final_dataset.csv:     {df_original.shape}')
print(f'new_agent_dataset.csv: {df_new.shape}')
print()
print('Group breakdown in original dataset:')
print(df_original['group'].value_counts().to_string())

# Align columns before concat (handles any ordering differences)
df_new_aligned = df_new.reindex(columns=df_original.columns)

df_merged = pd.concat([df_original, df_new_aligned], ignore_index=True)

# Deduplicate on the natural key — file_path must be included so that
# anonymous functions in different files of the same PR are not collapsed.
before    = len(df_merged)
df_merged = df_merged.drop_duplicates(
    subset=['repo', 'pull_request', 'file_path', 'function_name', 'function_start_line'],
    keep='first'
).reset_index(drop=True)

if before - len(df_merged):
    print(f'Dropped {before - len(df_merged)} duplicate rows.')

print()
print(f'Merged dataset shape: {df_merged.shape}')
print('Group breakdown in merged dataset:')
print(df_merged['group'].value_counts().to_string())
print('Label breakdown in merged dataset:')
print(df_merged['label'].value_counts().to_string())

df_merged.to_csv(output_path, index=False)
print(f'\nSaved to: {output_path}') 

final_dataset.csv:     (13917, 30)
new_agent_dataset.csv: (531, 30)

Group breakdown in original dataset:
group
agent    7239
human    6678
Dropped 9 duplicate rows.

Merged dataset shape: (14439, 30)
Group breakdown in merged dataset:
group
agent    7761
human    6678
Label breakdown in merged dataset:
label
Claude_Code     3242
Copilot         1687
Cursor          1247
Devin           1030
OpenAI_Codex     555

Saved to: G:\On the Naturalness of Agent-Generated Documentation\dataset\data\dev_agent_combined.csv


In [6]:
import pandas as pd
import numpy as np
import math
from collections import Counter
import re
import textstat


df = pd.read_csv(output_path)


def clean_comments(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # Single alternation scanned left-to-right (finditer) instead of three
    # independent findall passes over the same text. Three separate passes
    # can double-count: if a "#"/"//" comment's content happens to contain
    # something that looks like a \"\"\"...\"\"\" span (or vice versa), the
    # old code matched it in both passes and appended it twice. The inline
    # alternative also stops at a following triple-quote or /* block, not
    # just the next #/// or end of string -- otherwise a "#" comment right
    # next to a standalone docstring fragment swallows it whole, quote
    # markers included, instead of letting it be independently matched
    # and stripped.
    combined_pattern = re.compile(
        r'/\*+([\s\S]*?)\*/'
        r'|["\']{3}([\s\S]*?)["\']{3}'
        r'|(?:#|///|//)\s*(.*?)\s*(?=#|///|//|["\']{3}|/\*|$)'
    )

    cleaned_comments = []
    for m in combined_pattern.finditer(text):
        item = next(g for g in m.groups() if g is not None)

        # Remove the leading '*' from each line (common in JSDoc/C-style)
        clean_item = re.sub(r'^\s*\* ?', '', item, flags=re.MULTILINE)

        # Collapse newlines and tabs into a single space
        clean_item = ' '.join(clean_item.split())

        if clean_item.strip():
            cleaned_comments.append(clean_item.strip())

    return ' '.join(cleaned_comments)


for idx, row in df.iterrows():
    original_doc = row["doc_text"]
    code_text = row["function"]
    file_extension = _file_extension(row["file_path"])
    code_text_no_doc = strip_comments(code_text, file_extension)

    doc_text = clean_comments(original_doc)

    if isinstance(doc_text, str) and doc_text.strip():
        # Extraction found comment markers — use cleaned text and update doc_text
        df.at[idx, "doc_entropy"] = calculate_entropy(doc_text)
        df.at[idx, "doc_readability"] = textstat.flesch_reading_ease(doc_text)
        df.at[idx, "doc_code_overlap"] = doc_code_overlap(doc_text, code_text_no_doc)
        df.at[idx, "doc_redundancy"] = doc_redundancy(doc_text)
        df.at[idx, "doc_text"] = doc_text
    elif isinstance(original_doc, str) and original_doc.strip():
        # No markers found — doc_text is already plain prose, compute metrics from it directly
        df.at[idx, "doc_entropy"] = calculate_entropy(original_doc)
        df.at[idx, "doc_readability"] = textstat.flesch_reading_ease(original_doc)
        df.at[idx, "doc_code_overlap"] = doc_code_overlap(original_doc, code_text_no_doc)
        df.at[idx, "doc_redundancy"] = doc_redundancy(original_doc)
        # doc_text stays unchanged
    else:
        # Genuinely no documentation
        df.at[idx, "doc_entropy"] = np.nan
        df.at[idx, "doc_readability"] = np.nan
        df.at[idx, "doc_code_overlap"] = np.nan
        df.at[idx, "doc_redundancy"] = np.nan


# Save the updated dataset
df.to_csv(output_path, index=False)
print(f"Recomputed metrics and saved to {output_path}")

Recomputed metrics and saved to G:\On the Naturalness of Agent-Generated Documentation\dataset\data\dev_agent_combined.csv


In [7]:
# drop functions with no latin characters

import re

def _has_latin(text):
    return isinstance(text, str) and bool(re.search(r'[A-Za-z]', text))

before = len(df)
non_latin_documented = df['doc_text'].notna() & ~df['doc_text'].apply(_has_latin)
df = df[~non_latin_documented].reset_index(drop=True)
removed = before - len(df)

if removed:
    print(f'Dropped {removed} functions whose doc_text has zero Latin characters.')
else:
    print('No non-Latin-only documented functions found.')

df.to_csv(output_path, index=False)

Dropped 626 functions whose doc_text has zero Latin characters.
